# Bangla Halal/Haram Classification — Bangla-BERT

This is your `xyz` notebook's complete working pipeline (the full script from its last cell),
cleaned up into a linear, top-to-bottom notebook and re-pointed at `translated_dataset.csv`.

**Everything about the training pipeline itself — model, tokenizer, stratified split, custom
Trainer with history tracking, training args (cosine LR, label smoothing, gradient clipping,
early stopping), confusion matrices, training curves, classification reports, and the inference
function — is unchanged from your script.**

Only two things had to change, since the new dataset is shaped differently:
1. **Data loading** — `pd.read_csv("translated_dataset.csv")` instead of
   `pd.read_excel("BanglaBlendCleanedDataWithEnglishTranslation.xlsx")`, and columns
   `bangla` / `label` instead of `Sentence` / `Labels`.
2. **Label encoding** — your labels were already integers (`.astype(int)`); this dataset's
   labels are the strings `halal` / `haram`, so they're mapped to `0` / `1` first.

Run this on Colab/Kaggle with a GPU runtime (Runtime → Change runtime type → GPU).


## 1. Install dependencies

In [ ]:
!pip install -q transformers datasets torch scikit-learn pandas seaborn matplotlib

# torchvision (if preinstalled) is often a mismatched build in Colab/Kaggle images and isn't
# needed for text classification. `datasets` lazily tries `from torchvision.io import VideoReader`
# when tensorizing batches, and a broken/mismatched torchvision makes that raise ImportError
# instead of being skipped -- which crashes DataLoader worker processes during training.
# Removing it avoids that code path entirely.
!pip uninstall -y -q torchvision torchaudio


## 2. Imports, NumPy 2.0 monkey patch, device (unchanged from your script)

In [ ]:
import os
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
import numpy as np
import datasets.formatting.formatting
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)
from sklearn.model_selection import StratifiedShuffleSplit
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------------------------------
# Monkey patch for NumPy 2.0 error in datasets
def fixed_arrow_array_to_numpy(self, pa_array):
    array = pa_array.to_pandas().values
    return np.array(array, copy=True)

datasets.formatting.formatting.NumpyArrowExtractor._arrow_array_to_numpy = fixed_arrow_array_to_numpy
# -----------------------------------------------------

# Disable wandb
os.environ["WANDB_DISABLED"] = "true"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
DATA_PATH = "/content/drive/My Drive/translated_dataset.csv"  # Update this path if your file is in a different location

In [ ]:
# To avoid re-running the data loading and preprocessing in cell `153fee8b`, I'm going to rerun just this part.
df = pd.read_csv(DATA_PATH)
df = df.dropna()

# Keep only needed columns and rename (bangla -> text, label -> label)
df = df[["bangla", "label"]].copy()
df = df.rename(columns={"bangla": "text"})

# Map halal/haram -> 0/1 (your original data already had int labels; this one doesn't)
label2id = {"halal": 0, "haram": 1}
id2label = {v: k for k, v in label2id.items()}
df["label"] = df["label"].str.lower().map(label2id)
assert df["label"].isnull().sum() == 0, "Found labels other than halal/haram"
df["label"] = df["label"].astype(int)

# Basic text preprocessing (unchanged)
def preprocess_text(text):
    """Basic text preprocessing"""
    if pd.isna(text):
        return ""
    text = str(text).strip()
    text = ' '.join(text.split())
    return text

df["text"] = df["text"].apply(preprocess_text)

# Remove empty texts and duplicates (unchanged)
df = df[df["text"].str.len() > 0]
df = df.drop_duplicates(subset=["text"])

print("Columns after clean & rename:", df.columns)
print("Data shape:", df.shape)
print("Label distribution:")
print(df["label"].value_counts())
display(df.head())

## 3. Load & clean the dataset

Adapted: `translated_dataset.csv` (columns `text`, `label`, `bangla`) instead of the `.xlsx` file,
and we train on the `bangla` column. Labels (`halal`/`haram`) are mapped to `0`/`1` since the rest
of the pipeline expects integer labels.

In [ ]:
DATA_PATH = "translated_dataset.csv"  # <-- update path if needed (e.g. "/kaggle/input/.../translated_dataset.csv")

df = pd.read_csv(DATA_PATH)
df = df.dropna()

# Keep only needed columns and rename (bangla -> text, label -> label)
df = df[["bangla", "label"]].copy()
df = df.rename(columns={"bangla": "text"})

# Map halal/haram -> 0/1 (your original data already had int labels; this one doesn't)
label2id = {"halal": 0, "haram": 1}
id2label = {v: k for k, v in label2id.items()}
df["label"] = df["label"].str.lower().map(label2id)
assert df["label"].isnull().sum() == 0, "Found labels other than halal/haram"
df["label"] = df["label"].astype(int)

# Basic text preprocessing (unchanged)
def preprocess_text(text):
    """Basic text preprocessing"""
    if pd.isna(text):
        return ""
    text = str(text).strip()
    text = ' '.join(text.split())
    return text

df["text"] = df["text"].apply(preprocess_text)

# Remove empty texts and duplicates (unchanged)
df = df[df["text"].str.len() > 0]
df = df.drop_duplicates(subset=["text"])

print("Columns after clean & rename:", df.columns)
print("Data shape:", df.shape)
print("Label distribution:")
print(df["label"].value_counts())
df.head()


## 4. Load tokenizer & model (unchanged)

In [ ]:
model_name = "sagorsarker/bangla-bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    hidden_dropout_prob=0.5,
    attention_probs_dropout_prob=0.3,
    classifier_dropout=0.1
).to(device)


## 5. Tokenize & build dataset (unchanged)

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding=False,  # Dynamic padding in data collator
        truncation=True,
        max_length=256,
        return_tensors=None
    )

dataset = Dataset.from_pandas(df)
dataset = dataset.map(tokenize, batched=True, remove_columns=["text"])


## 6. Stratified train / validation / test split (unchanged)

In [ ]:
def create_stratified_splits(dataset, train_size=0.7, val_size=0.15, test_size=0.15, random_state=42):
    labels = [item['label'] for item in dataset]

    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=(val_size + test_size), random_state=random_state)
    train_idx, temp_idx = next(sss1.split(range(len(labels)), labels))

    temp_labels = [labels[i] for i in temp_idx]
    val_ratio = val_size / (val_size + test_size)
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=(1 - val_ratio), random_state=random_state)
    val_idx_temp, test_idx_temp = next(sss2.split(range(len(temp_idx)), temp_labels))

    val_idx = [temp_idx[i] for i in val_idx_temp]
    test_idx = [temp_idx[i] for i in test_idx_temp]

    return {
        'train': dataset.select(train_idx),
        'validation': dataset.select(val_idx),
        'test': dataset.select(test_idx)
    }

dataset_splits = create_stratified_splits(dataset)

for split in dataset_splits:
    dataset_splits[split].set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

print(f"Train size: {len(dataset_splits['train'])}")
print(f"Validation size: {len(dataset_splits['validation'])}")
print(f"Test size: {len(dataset_splits['test'])}")


## 7. Data collator & metrics (unchanged)

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    precision = precision_score(labels, preds, average="weighted")
    recall = recall_score(labels, preds, average="weighted")

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }


## 7b. K-Fold Cross-Validation (Stratified, K=5)

Runs `K_FOLDS` stratified folds over the full cleaned dataset (`df`) to get a more robust
estimate of performance than a single train/val/test split. Each fold trains a **fresh**
Bangla-BERT model from scratch for `num_train_epochs=20` (same hyperparameters used later
for the final model) and is evaluated on its own held-out fold.

This runs independently of, and *before*, the single stratified split + final training below
(section 8 onward), which still trains one model on the fixed train/val/test split and saves it.
Change `K_FOLDS` below if you want a different number of folds.

In [ ]:
from sklearn.model_selection import StratifiedKFold

K_FOLDS = 5  # <-- change number of folds here
CV_EPOCHS = 20

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

texts = df["text"].tolist()
cv_labels = df["label"].tolist()

fold_metrics = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(texts, cv_labels), start=1):
    print(f"\n===== Fold {fold_idx}/{K_FOLDS} =====")

    fold_train_df = df.iloc[train_idx].reset_index(drop=True)
    fold_val_df = df.iloc[val_idx].reset_index(drop=True)

    fold_train_ds = Dataset.from_pandas(fold_train_df).map(tokenize, batched=True, remove_columns=["text"])
    fold_val_ds = Dataset.from_pandas(fold_val_df).map(tokenize, batched=True, remove_columns=["text"])

    fold_train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    fold_val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

    fold_model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        hidden_dropout_prob=0.5,
        attention_probs_dropout_prob=0.3,
        classifier_dropout=0.1
    ).to(device)

    fold_args = TrainingArguments(
        output_dir=f"./results_fold{fold_idx}",
        num_train_epochs=CV_EPOCHS,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=2,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_dir=f"./logs_fold{fold_idx}",
        logging_steps=50,
        learning_rate=3e-5,
        lr_scheduler_type="cosine",
        warmup_steps=500,
        weight_decay=0.1,
        adam_epsilon=1e-6,
        max_grad_norm=1.0,
        label_smoothing_factor=0.1,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        save_total_limit=1,
        dataloader_num_workers=0,
        remove_unused_columns=False,
        report_to=[]
    )

    fold_trainer = Trainer(
        model=fold_model,
        args=fold_args,
        train_dataset=fold_train_ds,
        eval_dataset=fold_val_ds,
        compute_metrics=compute_metrics,
        data_collator=data_collator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    fold_trainer.train()
    metrics = fold_trainer.evaluate()
    metrics["fold"] = fold_idx
    fold_metrics.append(metrics)

    print(
        f"Fold {fold_idx} results: "
        f"accuracy={metrics['eval_accuracy']:.4f}, "
        f"f1={metrics['eval_f1']:.4f}, "
        f"precision={metrics['eval_precision']:.4f}, "
        f"recall={metrics['eval_recall']:.4f}"
    )

    # Free GPU memory before the next fold
    del fold_model, fold_trainer
    torch.cuda.empty_cache()


In [ ]:
cv_df = pd.DataFrame(fold_metrics)

print("\n===== K-Fold Cross-Validation Summary =====")
print(cv_df[["fold", "eval_accuracy", "eval_f1", "eval_precision", "eval_recall"]])

print("\nMean ± Std across folds:")
for metric in ["eval_accuracy", "eval_f1", "eval_precision", "eval_recall"]:
    mean = cv_df[metric].mean()
    std = cv_df[metric].std()
    print(f"{metric}: {mean:.4f} ± {std:.4f}")


## 8. Training arguments (unchanged)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=20,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,  # Effective batch size = 32
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,

    # Learning rate scheduling
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_steps=500,

    # Regularization
    weight_decay=0.1,
    adam_epsilon=1e-6,
    max_grad_norm=1.0,
    label_smoothing_factor=0.1,

    # Early stopping and model selection
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    # Evaluation and saving
    eval_steps=100,
    save_steps=200,
    save_total_limit=3,

    dataloader_num_workers=0,
    remove_unused_columns=False,
    report_to=[]
)


## 9. Custom Trainer that tracks history (unchanged)

In [ ]:
class CustomTrainer(Trainer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.train_losses = []
        self.val_losses = []
        self.val_accuracies = []
        self.epochs = []

    def log(self, logs):
        super().log(logs)
        if "train_loss" in logs:
            self.train_losses.append(logs["train_loss"])

        if "eval_loss" in logs:
            self.val_losses.append(logs["eval_loss"])
            self.val_accuracies.append(logs.get("eval_accuracy", 0))
            self.epochs.append(len(self.val_losses))

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_splits["train"],
    eval_dataset=dataset_splits["validation"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)


## 10. Train (unchanged)

In [ ]:
print("🚀 Starting training...")
trainer.train()


## 11. Evaluate + overfitting analysis (unchanged)

In [ ]:
print("\n📊 Validation Results:")
val_metrics = trainer.evaluate()
for key, value in val_metrics.items():
    print(f"{key}: {value:.4f}")

print("\n📊 Test Results:")
test_metrics = trainer.evaluate(dataset_splits["test"])
for key, value in test_metrics.items():
    print(f"test_{key}: {value:.4f}")

train_metrics = trainer.evaluate(dataset_splits["train"])
overfitting_accuracy = train_metrics['eval_accuracy'] - test_metrics['eval_accuracy']
overfitting_f1 = train_metrics['eval_f1'] - test_metrics['eval_f1']

print(f"\n🎯 Overfitting Analysis:")
print(f"Training Accuracy: {train_metrics['eval_accuracy']:.4f}")
print(f"Test Accuracy: {test_metrics['eval_accuracy']:.4f}")
print(f"Accuracy Gap (overfitting): {overfitting_accuracy:.4f}")
print(f"F1 Gap (overfitting): {overfitting_f1:.4f}")


## 12. Save the model (unchanged)

In [ ]:
model.save_pretrained("./final_bangla_bert_model")
tokenizer.save_pretrained("./final_bangla_bert_model")
print("\n✅ Best model saved at ./final_bangla_bert_model")


## 13. Confusion matrices & training curves

Unchanged, except `class_names` now shows `halal`/`haram` instead of generic `Class 0` / `Class 1`.

In [ ]:
def get_predictions(trainer, dataset):
    """Get predictions from the model"""
    predictions = trainer.predict(dataset)
    y_pred = predictions.predictions.argmax(-1)
    y_true = predictions.label_ids
    return y_true, y_pred

def plot_confusion_matrix(y_true, y_pred, class_names=None, title="Confusion Matrix"):
    """Plot confusion matrix"""
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()

    return cm

def plot_training_history(trainer):
    """Plot training and validation loss/accuracy over epochs"""
    epochs = range(1, len(trainer.val_losses) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    ax1.plot(epochs, trainer.val_losses, 'b-', label='Validation Loss', linewidth=2)
    if trainer.train_losses:
        train_epochs = np.linspace(1, len(trainer.val_losses), len(trainer.train_losses))
        ax1.plot(train_epochs, trainer.train_losses, 'r-', label='Training Loss', linewidth=2)

    ax1.set_title('Training & Validation Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, trainer.val_accuracies, 'g-', label='Validation Accuracy', linewidth=2)
    ax2.set_title('Validation Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

print("\n📊 Generating Confusion Matrices...")
class_names = [id2label[i] for i in sorted(df['label'].unique())]

val_true, val_pred = get_predictions(trainer, dataset_splits["validation"])
print(f"\nValidation Confusion Matrix:")
val_cm = plot_confusion_matrix(val_true, val_pred, class_names, "Validation Set Confusion Matrix")
print(val_cm)

test_true, test_pred = get_predictions(trainer, dataset_splits["test"])
print(f"\nTest Confusion Matrix:")
test_cm = plot_confusion_matrix(test_true, test_pred, class_names, "Test Set Confusion Matrix")
print(test_cm)

print("\n📈 Training History:")
plot_training_history(trainer)


## 14. Classification reports (unchanged)

In [ ]:
print("\n📊 Detailed Classification Report (Test Set):")
print(classification_report(test_true, test_pred, target_names=class_names))

print("\n📊 Detailed Classification Report (Validation Set):")
print(classification_report(val_true, val_pred, target_names=class_names))


## 15. Inference on new text

Unchanged, just swapped the sample sentence for one from your `bangla` column.

In [ ]:
def predict_text(text, model, tokenizer, device):
    """Function to predict on new text"""
    model.eval()
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_class = torch.argmax(predictions, dim=-1)

    return predicted_class.item(), predictions.cpu().numpy()

sample_text = df["text"].iloc[0]
try:
    pred_class, pred_probs = predict_text(sample_text, model, tokenizer, device)
    print(f"\nSample prediction:")
    print(f"Text: {sample_text}")
    print(f"Predicted class: {id2label[pred_class]}")
    print(f"Probabilities: {pred_probs}")
except Exception as e:
    print(f"Prediction error: {e}")
